Siddhartha 102303838 ASSIGNMENT-4

In [ ]:
#QUESTION 1
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin 

titles = []
prices = []
availability = []
star_ratings = []

current_url = "https://books.toscrape.com/catalogue/page-1.html"
base_url = "https://books.toscrape.com/catalogue/"

while current_url:
    
    web = requests.get(current_url).text
    soup = BeautifulSoup(web, "html.parser")
    books = soup.find_all("article", class_="product_pod")
    for book in books:
        # Title
        title = book.h3.a["title"]
        titles.append(title)
        
        # Price
        price = book.find("p", class_="price_color").text
        prices.append(price)
        
        # Availability
        avail_text = book.find("p", class_="instock availability").text.strip()
        availability.append(avail_text)
        
        # Star Rating
        star_class = book.find("p", class_="star-rating")["class"]
        star = [cls for cls in star_class if cls != "star-rating"][0]
        star_ratings.append(star)

    
    next_button = soup.find("li", class_="next")
    
    if next_button and next_button.a:
        
        next_page_relative_url = next_button.a["href"]
        
        current_url = urljoin(base_url, next_page_relative_url)
    else:
        current_url = None
df = pd.DataFrame({
    "Title": titles,
    "Price": prices,
    "Availability": availability,
    "Star Rating": star_ratings
})

df.info()
df.to_csv("books.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Title         1000 non-null   object
 1   Price         1000 non-null   object
 2   Availability  1000 non-null   object
 3   Star Rating   1000 non-null   object
dtypes: object(4)
memory usage: 31.4+ KB


In [ ]:
#QUESTION 2
import time
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By

DRIVER_PATH = 'chromedriver.exe' 
service = Service(executable_path=DRIVER_PATH)
driver = webdriver.Chrome(service=service)

url = "https://www.imdb.com/chart/top/"

driver.get(url)

last_height = driver.execute_script("return document.body.scrollHeight")

print("Scrolling down to load all movies...")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

    time.sleep(2)

    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

print("Finished loading the page.")

page_source = driver.page_source
soup = BeautifulSoup(page_source, "html.parser")

driver.quit()

movie_list_items = soup.find_all('li', class_='ipc-metadata-list-summary-item')

ranks = []
titles = []
years = []
ratings = []

for movie in movie_list_items:
    title_element = movie.find('h3', class_='ipc-title__text')
    if title_element:
        full_title_text = title_element.text.strip()
        rank_str, title_str = full_title_text.split('.', 1)
        ranks.append(rank_str.strip())
        titles.append(title_str.strip())
    else:
        ranks.append("N/A")
        titles.append("N/A")

    metadata_div = movie.find('div', class_='sc-15ac7568-6')
    if metadata_div:
        metadata_spans = metadata_div.find_all('span', class_='cli-title-metadata-item')
        if metadata_spans:
            years.append(metadata_spans[0].text.strip())
        else:
            years.append("N/A")
    else:
        years.append("N/A")

    rating_container = movie.find('div', attrs={'data-testid': 'ratingGroup--container'})
    if rating_container:
        rating_element = rating_container.find('span', class_='ipc-rating-star--rating')
        rating = rating_element.text.strip() if rating_element else "N/A"
        ratings.append(rating)
    else:
        ratings.append("N/A")

df = pd.DataFrame({
    "Rank": ranks,
    "Title": titles,
    "Year": years,
    "IMDb Rating": ratings
})

print(f"\nSuccessfully scraped {len(df)} movies.")
print(df)

df.to_csv("imdb_top_250.csv", index=False)

Scrolling down to load all movies...
Finished loading the page.

Successfully scraped 250 movies.
    Rank                     Title  Year IMDb Rating
0      1  The Shawshank Redemption  1994         9.3
1      2             The Godfather  1972         9.2
2      3           The Dark Knight  2008         9.1
3      4     The Godfather Part II  1974         9.0
4      5              12 Angry Men  1957         9.0
..   ...                       ...   ...         ...
245  246        Gangs of Wasseypur  2012         8.2
246  247             Into the Wild  2007         8.0
247  248                  The Help  2011         8.1
248  249             Groundhog Day  1993         8.0
249  250                  Drishyam  2015         8.2

[250 rows x 4 columns]


In [ ]:
#QUESTION 3
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = 'https://www.timeanddate.com/weather/'

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

response = requests.get(url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')

weather_data = []

main_table = soup.find('table', class_='zebra fw tb-theme')

if main_table:
    rows = main_table.find_all('tr')

    for row in rows[1:]:
        cells = row.find_all('td')
        
        for i in range(0, len(cells), 4):
            if i + 3 < len(cells):
                city_cell = cells[i]
                weather_cell = cells[i+2]
                temp_cell = cells[i+3]

                city_link = city_cell.find('a', href=True)
                city_name = city_link.text.strip() if city_link else 'N/A'

                img_tag = weather_cell.find('img')
                if img_tag and img_tag.has_attr('alt'):
                    weather_condition = img_tag['alt'].strip()
                else:
                    weather_condition = 'N/A'

                # Temperature
                temperature = temp_cell.text.strip() if temp_cell else 'N/A'

                # Append data
                weather_data.append([city_name, temperature, weather_condition])

df = pd.DataFrame(weather_data, columns=['City Name', 'Temperature', 'Weather Condition'])

print(f"Total cities extracted: {len(df)}")
print(df.head(10))
df.to_csv('weather_data.csv', index=False)


Total cities extracted: 140
      City Name Temperature                   Weather Condition
0         Accra       27 °C               Passing clouds. Warm.
1  Kuala Lumpur       28 °C               Passing clouds. Warm.
2   Addis Ababa       17 °C                 Partly sunny. Mild.
3   Kuwait City       41 °C      Passing clouds. Extremely hot.
4      Adelaide       14 °C                               Cool.
5          Kyiv       22 °C                     Overcast. Mild.
6       Algiers       26 °C                 Partly sunny. Warm.
7        La Paz       11 °C               Passing clouds. Cool.
8        Almaty       18 °C               Passing clouds. Mild.
9         Lagos       29 °C  Thunderstorms. Partly sunny. Warm.
